# 2D Darcy flux: mass balance over a control volume

A steady two-dimensional Darcy flux field (cm/hr) passes through a 1 cm cube of porous media. The question is how fast the
volumetric water content $\theta$ inside the cube changes. It is answered three ways, each more faithful to the field than the last:

1. **Point divergence**: evaluate $-\nabla\cdot\mathbf q$ at the cube's centroid.
2. **Face-center flux**: approximate each face's flow with the flux at its center (midpoint rule).
3. **Exact face integrals**: integrate the normal flux over each face analytically, and cross-check with the divergence theorem.

Every headline number is checked with an `assert`, so if this notebook runs top to bottom, the answers are right.

## 1. Setup

In [ ]:
import math
from fractions import Fraction

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.patches import FancyArrow, Patch, Rectangle
import plotly
import plotly.graph_objects as go
import scipy
from scipy.integrate import quad
from IPython.display import HTML, display

import riskplot
from riskplot import PlotConfig, RiskHeatmap, SurfaceRiskPlot, WaterfallChart

%matplotlib inline
%config InlineBackend.figure_format = "retina"
np.set_printoptions(linewidth=120, precision=2, suppress=True, floatmode="fixed")

for mod in (np, pd, scipy, matplotlib, plotly, riskplot):
    print(f"{mod.__name__:<11} {mod.__version__}")

IN_COLOR, OUT_COLOR, NET_COLOR = "#2166ac", "#b2182b", "#4d4d4d"


def check(name, value, expected, tol):
    # Assert a computed value against the expected answer, then print it.
    assert math.isclose(float(value), expected, rel_tol=0.0, abs_tol=tol), (
        f"{name}: got {float(value)!r}, expected {expected!r}"
    )
    print(f"  ok  {name:<34} {float(value):>12.4f}")

## 2. The flux field and its divergence

The flux has no $z$-component and no $z$-dependence:

$$
q_x(x,y) = 3x^2 + 5xy + 7y^2, \qquad
q_y(x,y) = -2x^3 + 4x^2y^3 - 2y \qquad [\mathrm{cm/hr}]
$$

With no sources or sinks, continuity (conservation of water volume) says storage changes only by net inflow:

$$
\frac{\partial \theta}{\partial t} = -\nabla\cdot\mathbf q = -\left(\frac{\partial q_x}{\partial x} + \frac{\partial q_y}{\partial y}\right),
\qquad
\frac{\partial q_x}{\partial x} = 6x + 5y,
\qquad
\frac{\partial q_y}{\partial y} = 12x^2y^2 - 2 .
$$

Each polynomial is stored as its coefficients `{(i, j): c}`, meaning $c\,x^i y^j$. The same objects can be evaluated on
numpy grids *and* differentiated or integrated exactly in rational arithmetic, so the derivatives above are computed, not typed in.

In [ ]:
QX = {(2, 0): 3, (1, 1): 5, (0, 2): 7}      # 3x^2 + 5xy + 7y^2
QY = {(3, 0): -2, (2, 3): 4, (0, 1): -2}    # -2x^3 + 4x^2 y^3 - 2y


def evaluate(poly, x, y):
    return sum(c * x**i * y**j for (i, j), c in poly.items())


def d_dx(poly):
    return {(i - 1, j): c * i for (i, j), c in poly.items() if i > 0}


def d_dy(poly):
    return {(i, j - 1): c * j for (i, j), c in poly.items() if j > 0}


DQX_DX = d_dx(QX)
DQY_DY = d_dy(QY)
assert DQX_DX == {(1, 0): 6, (0, 1): 5}        # 6x + 5y
assert DQY_DY == {(2, 2): 12, (0, 0): -2}      # 12x^2 y^2 - 2


def qx(x, y):
    return evaluate(QX, x, y)


def qy(x, y):
    return evaluate(QY, x, y)


def div_q(x, y):
    return evaluate(DQX_DX, x, y) + evaluate(DQY_DY, x, y)


# Sanity check the symbolic derivatives against central differences at random points.
rng = np.random.default_rng(0)
pts = rng.uniform([1.5, 2.5], [3.5, 4.5], size=(20, 2))
eps = 1e-6
fd_div = np.array([(qx(x + eps, y) - qx(x - eps, y)) / (2 * eps)
                   + (qy(x, y + eps) - qy(x, y - eps)) / (2 * eps) for x, y in pts])
assert np.allclose(fd_div, div_q(pts[:, 0], pts[:, 1]), rtol=1e-7)

# Control volume: x in [2, 3], y in [3, 4], dz = 1 cm
X0, X1, Y0, Y1, DZ = 2, 3, 3, 4, 1
XC, YC = (X0 + X1) / 2, (Y0 + Y1) / 2
V = (X1 - X0) * (Y1 - Y0) * DZ
A_X = (Y1 - Y0) * DZ      # area of the left/right faces
A_Y = (X1 - X0) * DZ      # area of the bottom/top faces
print(f"V = {V} cm^3, face areas = {A_X}, {A_Y} cm^2, centroid = ({XC}, {YC})")

## 3. The field as matrices

`np.meshgrid(x, y)` returns two matrices with `X[i, j] = x[j]` and `Y[i, j] = y[i]`. Any field evaluated on them is a matrix
whose **row $i$ is $y_i$** and **column $j$ is $x_j$**. The coarse 0.5 cm grid below deliberately contains the box corners,
the four face centers, and the centroid. The matrices hold every number Methods 1 and 2 need.

In [ ]:
xc = np.arange(1.5, 3.5 + 1e-9, 0.5)
yc = np.arange(2.5, 4.5 + 1e-9, 0.5)
Xc, Yc = np.meshgrid(xc, yc)

QXc = qx(Xc, Yc)
QYc = qy(Xc, Yc)
MAGc = np.hypot(QXc, QYc)
DIVc = div_q(Xc, Yc)

print("x nodes:", xc)
print("y nodes:", yc)
coarse = {"X": Xc, "Y": Yc, "q_x [cm/hr]": QXc, "q_y [cm/hr]": QYc, "|q| [cm/hr]": MAGc, "div q [1/hr]": DIVc}
for name, M in coarse.items():
    print(f"\n{name}   shape={M.shape}   (row i <-> y[i], column j <-> x[j])")
    print(M)

The same matrices as heatmaps, drawn with RiskPlot's `RiskHeatmap`. Rows are flipped so $y$ increases upward. The white
outline marks the 3 × 3 block of nodes that lie on or inside the control volume: its corners, face centers, and centroid.

Read the box straight off the matrices. In the $y=3.5$ row, $q_x$ goes from 132.75 (left face center) to 165.25 (right face center),
a gentle change. In the $x=2.5$ column, $q_y$ jumps from 637.75 (bottom) to 1560.75 (top), roughly 2.4×. Almost all of the
imbalance, and all of the disagreement between methods, is in $y$.

In [ ]:
def matrix_heatmap(M, title, units, cmap):
    df = pd.DataFrame(M[::-1], index=[f"{v:.1f}" for v in yc[::-1]], columns=[f"{v:.1f}" for v in xc])
    fig, ax = RiskHeatmap(PlotConfig(grid=False)).plot(
        df, annot=True, fmt=".1f", cmap=cmap, figsize=(6.4, 5.2), title=title
    )
    # RiskHeatmap chooses white/black annotation text with a |value| > 0.5 rule meant for
    # correlation matrices. Every value here exceeds that, so recolor by cell brightness.
    im = ax.images[0]
    for t in ax.texts:
        j, i = t.get_position()
        r, g, b, _ = im.cmap(im.norm(df.values[int(round(i)), int(round(j))]))
        t.set_color("black" if 0.299 * r + 0.587 * g + 0.114 * b > 0.5 else "white")
        t.set_fontsize(9)
    # Nodes on/inside the box are columns 1-3 and rows 1-3 of the flipped matrix; outline those cells.
    ax.add_patch(Rectangle((0.5, 0.5), 3, 3, fill=False, edgecolor="white", linewidth=3.5))
    ax.set_xlabel("x [cm]")
    ax.set_ylabel("y [cm]")
    fig.axes[-1].set_ylabel(units, rotation=270, labelpad=15)
    return fig, ax


matrix_heatmap(QXc, r"$q_x$ matrix", "cm/hr", "viridis")
matrix_heatmap(QYc, r"$q_y$ matrix", "cm/hr", "viridis")
matrix_heatmap(MAGc, r"$|\mathbf{q}|$ matrix", "cm/hr", "cividis")
matrix_heatmap(DIVc, r"$\nabla\cdot\mathbf{q}$ matrix", "1/hr", "magma")
plt.show()

## 4. Fine-grid fields

On a 161 × 161 grid the fields become smooth surfaces. RiskPlot's `SurfaceRiskPlot` (contourf mode) takes the fields in long
format and grids them back onto the same nodes. $|\mathbf q|$ is shown on a $\log_{10}$ scale because $q_y$ grows like $y^3$.

In [ ]:
xf = np.linspace(1.5, 3.5, 161)
yf = np.linspace(2.5, 4.5, 161)
Xf, Yf = np.meshgrid(xf, yf)
QXf, QYf = qx(Xf, Yf), qy(Xf, Yf)
MAGf = np.hypot(QXf, QYf)
LOGf = np.log10(MAGf)
DIVf = div_q(Xf, Yf)

fields = pd.DataFrame({
    "x": Xf.ravel(), "y": Yf.ravel(),
    "q_x": QXf.ravel(), "q_y": QYf.ravel(),
    "log10_mag": LOGf.ravel(), "div_q": DIVf.ravel(),
})
print(f"|q| on the plotting window spans {MAGf.min():.1f} to {MAGf.max():.1f} cm/hr "
      f"({np.log10(MAGf.max() / MAGf.min()):.2f} decades)")
print(f"q_y spans {QYf.min():.1f} to {QYf.max():.1f} cm/hr;  q_x spans {QXf.min():.1f} to {QXf.max():.1f} cm/hr")


def field_contour(column, title, units, cmap):
    cfg = PlotConfig(figsize=(6.4, 5.2), colormap=cmap, grid=False)
    fig, ax = SurfaceRiskPlot(cfg).plot(
        fields, "x", "y", column, surface_type="contourf", grid_resolution=len(xf),
        title=title, x_label="x [cm]", y_label="y [cm]",
    )
    ax.add_patch(Rectangle((X0, Y0), X1 - X0, Y1 - Y0, fill=False, edgecolor="white", linewidth=2, linestyle="--"))
    ax.set_aspect("equal")
    fig.axes[-1].set_ylabel(units, rotation=270, labelpad=15)
    fig.tight_layout()
    return fig, ax


field_contour("q_x", r"$q_x(x,y)$", "cm/hr", "viridis")
field_contour("q_y", r"$q_y(x,y)$", "cm/hr", "viridis")
field_contour("log10_mag", r"$\log_{10}|\mathbf{q}|$", r"$\log_{10}$ cm/hr", "cividis")
field_contour("div_q", r"$\nabla\cdot\mathbf{q}$", "1/hr", "magma")
plt.show()

An interactive view of $q_y$, from RiskPlot's plotly surface. Drag to rotate. The black curve traces $q_y$ around the
control volume's boundary; the climb from the bottom face ($y=3$) to the top face ($y=4$) is the $4x^2y^3$ term at work.

In [ ]:
coarse_fields = fields[(fields.index % 4 == 0)]    # thin the long table; the surface is re-gridded to 50x50 anyway
surf = SurfaceRiskPlot().plot_interactive(coarse_fields, "x", "y", "q_y", title="q_y(x, y) [cm/hr]")

s = np.linspace(0, 1, 50)
bx = np.concatenate([X0 + s, np.full_like(s, X1), X1 - s, np.full_like(s, X0)])
by = np.concatenate([np.full_like(s, Y0), Y0 + s, np.full_like(s, Y1), Y1 - s])
surf.add_trace(go.Scatter3d(x=bx, y=by, z=qy(bx, by), mode="lines", line=dict(color="black", width=6),
                            name="control volume boundary"))
surf.update_traces(selector=dict(type="surface"), colorscale="Viridis", colorbar=dict(title="cm/hr"))
surf.update_layout(width=760, height=620, margin=dict(l=0, r=0, t=50, b=0),
                   scene=dict(xaxis_title="x [cm]", yaxis_title="y [cm]", zaxis_title="q_y [cm/hr]"))
# Embed with a CDN script tag so the figure renders in the static nbconvert HTML page too.
display(HTML(surf.to_html(include_plotlyjs="cdn", full_html=False)))

## 5. Flow visualization

Because $|\mathbf q|$ varies by more than an order of magnitude, linearly scaled arrows would shrink to dots in the lower
left and overwhelm the upper right. Arrows here keep the true **direction** $\hat{\mathbf q} = \mathbf q/|\mathbf q|$, and their
**length** grows with $\log_{10}|\mathbf q|$:

$$
\ell = \ell_\text{max}\,\frac{\log_{10}|\mathbf q| - L_\text{min} + 0.3}{L_\text{max} - L_\text{min} + 0.3},
$$

where $L_\text{min}, L_\text{max}$ are the extremes of $\log_{10}|\mathbf q|$ over the window.

The first figure puts streamlines (line width also $\propto\log_{10}|\mathbf q|$) over the RiskPlot contour of $\log_{10}|\mathbf q|$.
The second shows the arrow field, with the flux at points along each face of the control volume highlighted.

In [ ]:
L_MIN, L_MAX = LOGf.min(), LOGf.max()
ARROW_MAX = 0.18   # cm, in plot coordinates


def log_scaled(u, v):
    mag = np.hypot(u, v)
    frac = (np.log10(mag) - L_MIN + 0.3) / (L_MAX - L_MIN + 0.3)
    return ARROW_MAX * frac * u / mag, ARROW_MAX * frac * v / mag


def draw_control_volume(ax, color, lw=2.5):
    ax.add_patch(Rectangle((X0, Y0), X1 - X0, Y1 - Y0, fill=False, edgecolor=color, linewidth=lw, zorder=5))
    for (x, y, label, ha, va) in [
        (X0 - 0.04, YC, "left\n(in)", "right", "center"), (X1 + 0.04, YC, "right\n(out)", "left", "center"),
        (XC, Y0 - 0.04, "bottom (in)", "center", "top"), (XC, Y1 + 0.04, "top (out)", "center", "bottom"),
    ]:
        ax.text(x, y, label, ha=ha, va=va, fontsize=9, color=color, fontweight="bold", zorder=6,
                bbox=dict(facecolor="white", alpha=0.75, edgecolor="none", pad=1.5))


# Background: RiskPlot contourf of log10|q|
cfg = PlotConfig(figsize=(8.5, 7.5), colormap="cividis", grid=False)
fig, ax = SurfaceRiskPlot(cfg).plot(
    fields, "x", "y", "log10_mag", surface_type="contourf", grid_resolution=len(xf),
    title=r"Streamlines over $\log_{10}|\mathbf{q}|$", x_label="x [cm]", y_label="y [cm]",
)
fig.axes[-1].set_ylabel(r"$\log_{10}|\mathbf{q}|$  [cm/hr]", rotation=270, labelpad=18)
# Streamlines: RiskPlot has no vector-field plot, so this overlay is plain matplotlib.
lw = 0.4 + 2.0 * (LOGf - L_MIN) / (L_MAX - L_MIN)
ax.streamplot(xf, yf, QXf, QYf, color="white", linewidth=lw, density=1.3, arrowsize=1.0)
draw_control_volume(ax, "#d7301f", lw=3)
ax.set_xlim(1.5, 3.5)
ax.set_ylim(2.5, 4.5)
ax.set_aspect("equal")
fig.tight_layout()
plt.show()

In [ ]:
# Quiver with per-face highlighting: no RiskPlot equivalent for vector arrows, so matplotlib.
xq = np.arange(1.625, 3.5, 0.25)
yq = np.arange(2.625, 4.5, 0.25)
Xq, Yq = np.meshgrid(xq, yq)
# Leave the node row/column just outside each face empty; the highlighted face arrows go there.
in_x, in_y = (Xq > X0) & (Xq < X1), (Yq > Y0) & (Yq < Y1)
near_face = ((np.isclose(Xq, X0 - 0.125) | np.isclose(Xq, X1 + 0.125)) & in_y) | \
            ((np.isclose(Yq, Y0 - 0.125) | np.isclose(Yq, Y1 + 0.125)) & in_x)
Xq, Yq = Xq[~near_face], Yq[~near_face]
Uq, Vq = log_scaled(qx(Xq, Yq), qy(Xq, Yq))

fig, ax = plt.subplots(figsize=(8.5, 7.5))
qv = ax.quiver(Xq, Yq, Uq, Vq, np.log10(np.hypot(qx(Xq, Yq), qy(Xq, Yq))), cmap="cividis",
               angles="xy", scale_units="xy", scale=1, pivot="middle", width=0.004, alpha=0.7)
cb = fig.colorbar(qv, ax=ax)
cb.set_label(r"$\log_{10}|\mathbf{q}|$  [cm/hr]", rotation=270, labelpad=18)

# Flux at four points along each face. Inflow arrows end on the face; outflow arrows start on it.
t = np.array([0.125, 0.375, 0.625, 0.875])
faces = {
    "left":   (np.full_like(t, X0), Y0 + t, "tip",  IN_COLOR),
    "bottom": (X0 + t, np.full_like(t, Y0), "tip",  IN_COLOR),
    "right":  (np.full_like(t, X1), Y0 + t, "tail", OUT_COLOR),
    "top":    (X0 + t, np.full_like(t, Y1), "tail", OUT_COLOR),
}
for name, (fx, fy, pivot, color) in faces.items():
    u, v = qx(fx, fy), qy(fx, fy)
    # Physical check: flow is +x and +y everywhere on the box, so these faces really are in/out.
    assert np.all(u > 0) and np.all(v > 0), name
    su, sv = log_scaled(u, v)
    ax.quiver(fx, fy, su, sv, color=color, angles="xy", scale_units="xy", scale=1,
              pivot=pivot, width=0.007, zorder=7)
ax.add_patch(Rectangle((X0, Y0), X1 - X0, Y1 - Y0, fill=False, edgecolor="black", linewidth=2.5, zorder=5))
ax.legend(handles=[Line2D([], [], color=IN_COLOR, lw=3, label="flux entering (left, bottom faces)"),
                   Line2D([], [], color=OUT_COLOR, lw=3, label="flux leaving (right, top faces)")],
          loc="upper center", bbox_to_anchor=(0.5, -0.08), ncol=2, frameon=False)
ax.set(xlim=(1.5, 3.5), ylim=(2.5, 4.5), xlabel="x [cm]", ylabel="y [cm]",
       title="Log-scaled flux vectors and the control volume")
ax.set_aspect("equal")
fig.tight_layout()
plt.show()

## 6. Method 1: point divergence at the centroid

Treat the cube as a single point and evaluate continuity at its centroid $(2.5, 3.5)$:

$$
\frac{\partial\theta}{\partial t}\bigg|_{(2.5,\,3.5)}
= -\big[\,6(2.5) + 5(3.5)\,\big] - \big[\,12(2.5)^2(3.5)^2 - 2\,\big]
= -32.50 - 916.75 = -949.25\ \mathrm{hr^{-1}}
$$

In [ ]:
m1_x = -evaluate(DQX_DX, XC, YC)
m1_y = -evaluate(DQY_DY, XC, YC)
m1_rate = m1_x + m1_y

print("Method 1: point divergence")
check("x-contribution [1/hr]", m1_x, -32.50, 1e-12)
check("y-contribution [1/hr]", m1_y, -916.75, 1e-12)
check("dtheta/dt [1/hr]", m1_rate, -949.25, 1e-12)

## 7. Method 2: face-center flux (midpoint rule)

Take the normal flux at each face's center as representative of the whole face, so $Q_f \approx q_n(\text{center})\,A_f$.
For the control volume, storage changes by net inflow:

$$
V\,\frac{\Delta\theta}{\Delta t} = Q_\text{in} - Q_\text{out}
= \big[q_x(2,3.5) + q_y(2.5,3)\big]A - \big[q_x(3,3.5) + q_y(2.5,4)\big]A
$$

In [ ]:
m2_Q = {
    "left":   qx(X0, YC) * A_X,
    "bottom": qy(XC, Y0) * A_Y,
    "right":  qx(X1, YC) * A_X,
    "top":    qy(XC, Y1) * A_Y,
}
m2_in = m2_Q["left"] + m2_Q["bottom"]
m2_out = m2_Q["right"] + m2_Q["top"]
m2_rate = (m2_in - m2_out) / V
m2_x = (m2_Q["left"] - m2_Q["right"]) / V
m2_y = (m2_Q["bottom"] - m2_Q["top"]) / V

display(pd.DataFrame({"face center (x, y)": [(X0, YC), (XC, Y0), (X1, YC), (XC, Y1)],
                      "q_n [cm/hr]": list(m2_Q.values()),
                      "Q [cm^3/hr]": list(m2_Q.values())}, index=list(m2_Q)).round(4))

print("Method 2: face-center flux")
check("Q_in [cm^3/hr]", m2_in, 770.50, 1e-12)
check("Q_out [cm^3/hr]", m2_out, 1726.00, 1e-12)
check("x-contribution [1/hr]", m2_x, -32.50, 1e-12)
check("dtheta/dt [1/hr]", m2_rate, -955.50, 1e-12)

## 8. Method 3: exact face integrals

Integrate the normal flux over each face ($dz$ integrates to 1 cm). The integrands are polynomials, so the results are exact rationals:

$$
\begin{aligned}
Q_\text{left}   &= \int_3^4 q_x(2,y)\,dy = \int_3^4 (12 + 10y + 7y^2)\,dy = \tfrac{400}{3} \approx 133.3333 \\
Q_\text{bottom} &= \int_2^3 q_y(x,3)\,dx = \int_2^3 (-2x^3 + 108x^2 - 6)\,dx = \tfrac{1291}{2} = 645.5 \\
Q_\text{right}  &= \int_3^4 q_x(3,y)\,dy = \int_3^4 (27 + 15y + 7y^2)\,dy = \tfrac{995}{6} \approx 165.8333 \\
Q_\text{top}    &= \int_2^3 q_y(x,4)\,dx = \int_2^3 (-2x^3 + 256x^2 - 8)\,dx = \tfrac{9485}{6} \approx 1580.8333
\end{aligned}
$$

The **divergence theorem** gives an independent route to the same net outflow:

$$
\oint_{\partial V} \mathbf q\cdot\hat{\mathbf n}\,dA = \int_V \nabla\cdot\mathbf q\,dV
= \int_2^3\!\!\int_3^4 \big(6x + 5y + 12x^2y^2 - 2\big)\,dy\,dx = \tfrac{65}{2} + \tfrac{2806}{3} = \tfrac{5807}{6} \approx 967.8333
$$

Below, the integrals are computed in exact `Fraction` arithmetic. They are asserted equal to the volume integral exactly, and to
`scipy.integrate.quad` numerically, which checks the integrator itself.

In [ ]:
def mono_int(n, a, b):
    # exact integral of t**n from a to b
    a, b = Fraction(a), Fraction(b)
    return (b ** (n + 1) - a ** (n + 1)) / (n + 1)


def face_integral_at_x(poly, x_face, y_lo, y_hi):
    return sum(c * Fraction(x_face) ** i * mono_int(j, y_lo, y_hi) for (i, j), c in poly.items())


def face_integral_at_y(poly, y_face, x_lo, x_hi):
    return sum(c * mono_int(i, x_lo, x_hi) * Fraction(y_face) ** j for (i, j), c in poly.items())


def volume_integral(poly, x_lo, x_hi, y_lo, y_hi, dz=1):
    return dz * sum(c * mono_int(i, x_lo, x_hi) * mono_int(j, y_lo, y_hi) for (i, j), c in poly.items())


m3_Q = {
    "left":   face_integral_at_x(QX, X0, Y0, Y1) * DZ,
    "bottom": face_integral_at_y(QY, Y0, X0, X1) * DZ,
    "right":  face_integral_at_x(QX, X1, Y0, Y1) * DZ,
    "top":    face_integral_at_y(QY, Y1, X0, X1) * DZ,
}
m3_in = m3_Q["left"] + m3_Q["bottom"]
m3_out = m3_Q["right"] + m3_Q["top"]
m3_rate = (m3_in - m3_out) / V
m3_x = (m3_Q["left"] - m3_Q["right"]) / V
m3_y = (m3_Q["bottom"] - m3_Q["top"]) / V
vol_div = volume_integral(DQX_DX, X0, X1, Y0, Y1, DZ) + volume_integral(DQY_DY, X0, X1, Y0, Y1, DZ)

assert m3_Q == {"left": Fraction(400, 3), "bottom": Fraction(1291, 2),
                "right": Fraction(995, 6), "top": Fraction(9485, 6)}
assert vol_div == m3_out - m3_in == Fraction(5807, 6)      # divergence theorem, exactly

# Independent numerical check of the face integrals
quad_Q = {
    "left":   quad(lambda y: qx(X0, y), Y0, Y1)[0] * DZ,
    "bottom": quad(lambda x: qy(x, Y0), X0, X1)[0] * DZ,
    "right":  quad(lambda y: qx(X1, y), Y0, Y1)[0] * DZ,
    "top":    quad(lambda x: qy(x, Y1), X0, X1)[0] * DZ,
}
for face in m3_Q:
    assert math.isclose(quad_Q[face], m3_Q[face], rel_tol=1e-12), face

display(pd.DataFrame({"exact": [str(v) for v in m3_Q.values()],
                      "Q [cm^3/hr]": [float(v) for v in m3_Q.values()],
                      "scipy quad": list(quad_Q.values())}, index=list(m3_Q)).round(4))

print("Method 3: exact face integrals")
check("Q_left [cm^3/hr]", m3_Q["left"], 133.3333, 5e-5)
check("Q_bottom [cm^3/hr]", m3_Q["bottom"], 645.5, 1e-12)
check("Q_right [cm^3/hr]", m3_Q["right"], 165.8333, 5e-5)
check("Q_top [cm^3/hr]", m3_Q["top"], 1580.8333, 5e-5)
check("Q_in [cm^3/hr]", m3_in, 778.8333, 5e-5)
check("Q_out [cm^3/hr]", m3_out, 1746.6667, 5e-5)
check("x-contribution [1/hr]", m3_x, -32.50, 1e-12)
check("dtheta/dt [1/hr]", m3_rate, -967.8333, 5e-5)
check("volume integral of div q", vol_div, 967.8333, 5e-5)
check("div theorem mismatch", vol_div - (m3_out - m3_in), 0.0, 0.0)

## 9. Why the methods disagree

Split each estimate into its $x$ and $y$ parts:

* **$x$: all three agree at −32.50 hr⁻¹.** $\partial q_x/\partial x = 6x + 5y$ is linear, so its average over the box equals its
  centroid value. Likewise $q_x(3,y) - q_x(2,y) = 15 + 5y$ is linear in $y$, so the midpoint rule integrates it exactly.
* **$y$: the $4x^2y^3$ term carries all of the disagreement.**
  * Method 1 → 2. The finite difference of $y^3$ across the box, $(4^3 - 3^3)/1 = 37$, exceeds the point derivative $3y_c^2 = 36.75$.
    That adds $0.25 \times 4x_c^2 = 6.25$.
  * Method 2 → 3. The average of $x^2$ along the top and bottom faces, $\overline{x^2} = x_c^2 + \tfrac{1}{12}$, exceeds $x_c^2$.
    That adds $4 \times 37 \times \tfrac{1}{12} = 12.33$.

In [ ]:
summary = pd.DataFrame(
    {
        "x-part [1/hr]": [m1_x, m2_x, float(m3_x)],
        "y-part [1/hr]": [m1_y, m2_y, float(m3_y)],
        "dtheta/dt [1/hr]": [m1_rate, m2_rate, float(m3_rate)],
        "Q_in [cm^3/hr]": [np.nan, m2_in, float(m3_in)],
        "Q_out [cm^3/hr]": [np.nan, m2_out, float(m3_out)],
    },
    index=["1. point divergence", "2. face-center flux", "3. exact face integrals"],
)
display(summary.round(4))

assert m1_x == m2_x == m3_x == -32.5
check("y-gap, method 1 -> 2", m1_y - m2_y, 6.25, 1e-12)
check("y-gap, method 2 -> 3", m2_y - float(m3_y), 37 / 3, 1e-9)

## 10. Mass balance on the control volume

The first figure sizes each face's arrow by its exact integrated flow $Q$ (midpoint-rule value in parentheses). The second is
a RiskPlot `WaterfallChart` budget: inflows add, outflows subtract, and the closing bar is the net change in stored water,
$V\,\partial\theta/\partial t$.

In [ ]:
# Face-flux diagram: RiskPlot has no annotated arrow/schematic plot, so this is matplotlib.
fig, ax = plt.subplots(figsize=(8, 8))
ax.add_patch(Rectangle((X0, Y0), 1, 1, facecolor="#f0f0f0", edgecolor="black", linewidth=2.5, zorder=2))

Q_max = float(max(m3_Q.values()))
W_MAX, LENGTH = 0.42, 0.62


def flux_arrow(x, y, dx, dy, Q, color):
    w = W_MAX * float(Q) / Q_max
    ax.add_patch(FancyArrow(x, y, dx, dy, width=w, head_width=w + 0.12, head_length=0.16,
                            length_includes_head=True, color=color, zorder=3))


gap = 0.03
flux_arrow(X0 - LENGTH - gap, YC, LENGTH, 0, m3_Q["left"], IN_COLOR)
flux_arrow(XC, Y0 - LENGTH - gap, 0, LENGTH, m3_Q["bottom"], IN_COLOR)
flux_arrow(X1 + gap, YC, LENGTH, 0, m3_Q["right"], OUT_COLOR)
flux_arrow(XC, Y1 + gap, 0, LENGTH, m3_Q["top"], OUT_COLOR)

label = dict(fontsize=11, ha="center", va="center", zorder=4)
ax.text(X0 - 0.36, YC + 0.2, f"left\n{float(m3_Q['left']):.2f}\n({m2_Q['left']:.2f})", color=IN_COLOR, **label)
ax.text(XC - 0.42, Y0 - 0.34, f"bottom\n{float(m3_Q['bottom']):.2f}\n({m2_Q['bottom']:.2f})", color=IN_COLOR, **label)
ax.text(X1 + 0.36, YC + 0.2, f"right\n{float(m3_Q['right']):.2f}\n({m2_Q['right']:.2f})", color=OUT_COLOR, **label)
ax.text(XC + 0.44, Y1 + 0.34, f"top\n{float(m3_Q['top']):.2f}\n({m2_Q['top']:.2f})", color=OUT_COLOR, **label)
ax.text(XC, YC,
        f"$Q_{{in}}$ = {float(m3_in):.2f}\n$Q_{{out}}$ = {float(m3_out):.2f}\n\n"
        f"$\\partial\\theta/\\partial t$ = {float(m3_rate):.2f} hr$^{{-1}}$",
        fontsize=12, ha="center", va="center", zorder=4)

ax.set(xlim=(1.2, 3.8), ylim=(2.2, 4.8), title="Integrated face flows  Q [cm$^3$/hr]   exact (midpoint)")
ax.set_aspect("equal")
ax.axis("off")
fig.tight_layout()
plt.show()

In [ ]:
budget = pd.DataFrame({
    "term": ["left face (in)", "bottom face (in)", "right face (out)", "top face (out)"],
    "Q": [float(m3_Q["left"]), float(m3_Q["bottom"]), -float(m3_Q["right"]), -float(m3_Q["top"])],
})
net = budget["Q"].sum()
check("budget net = V dtheta/dt", net, -967.8333, 5e-5)

fig, ax = WaterfallChart(PlotConfig(figsize=(9, 6))).plot(
    budget, "term", "Q", positive_color=IN_COLOR, negative_color=OUT_COLOR, total_color=NET_COLOR,
    value_format="{:+.2f}", title="Control-volume water budget (exact face integrals)",
)
# WaterfallChart draws the closing "Total" bar as abs(value) upward from zero, which would show
# this net loss as a gain. Move that bar and its label below the axis where the value belongs.
total_bar, total_text = ax.patches[-1], ax.texts[-1]
total_bar.set_y(net)
total_text.set_y(net / 2)
total_text.set_text(f"net: {net:+.2f}")
ax.set_xticks(ax.get_xticks(), [*budget["term"], "net = V dθ/dt"], rotation=20, ha="right")
ax.legend(handles=[Patch(facecolor=IN_COLOR, label="inflow"), Patch(facecolor=OUT_COLOR, label="outflow"),
                   Patch(facecolor=NET_COLOR, label="net")], loc="lower left")
ax.set_ylabel(r"Q  [cm$^3$/hr]")
ax.set_ylim(net * 1.12, float(m3_in) * 1.15)
fig.tight_layout()
plt.show()

## 11. Convergence as the box shrinks

Shrink the box to half-width $h$ around the centroid (side $2h$). The face-center estimate becomes

$$
\frac{\partial\theta}{\partial t}\bigg|_\text{faces}(h)
= -\frac{q_x(x_c+h,y_c) - q_x(x_c-h,y_c)}{2h} - \frac{q_y(x_c,y_c+h) - q_y(x_c,y_c-h)}{2h}.
$$

These are central differences. They are exact for the quadratic $q_x$, and for $q_y$ only the $y^3$ term contributes error, since
$\frac{(y+h)^3-(y-h)^3}{2h} = 3y^2 + h^2$. So

$$
\frac{\partial\theta}{\partial t}\bigg|_\text{faces}(h) = -949.25 - 4x_c^2h^2 = -949.25 - 25h^2,
$$

which converges to the point divergence at second order. At $h = 0.5$ it recovers −955.50. The exact face integral (volume average of
$-\nabla\cdot\mathbf q$) also converges quadratically, with a larger constant: $-949.25 - 74h^2 - \tfrac{4}{3}h^4$.

In [ ]:
h = np.logspace(np.log10(0.5), -3, 30)


def face_center_rate(h):
    side = 2 * h
    q_in = qx(XC - h, YC) * side * DZ + qy(XC, YC - h) * side * DZ
    q_out = qx(XC + h, YC) * side * DZ + qy(XC, YC + h) * side * DZ
    return (q_in - q_out) / (side * side * DZ)


def exact_rate(h):
    hf, xc_, yc_ = Fraction(h), Fraction(XC), Fraction(YC)
    x_lo, x_hi, y_lo, y_hi = xc_ - hf, xc_ + hf, yc_ - hf, yc_ + hf
    q_in = face_integral_at_x(QX, x_lo, y_lo, y_hi) + face_integral_at_y(QY, y_lo, x_lo, x_hi)
    q_out = face_integral_at_x(QX, x_hi, y_lo, y_hi) + face_integral_at_y(QY, y_hi, x_lo, x_hi)
    return float((q_in - q_out) * DZ / ((x_hi - x_lo) * (y_hi - y_lo) * DZ))


fc = face_center_rate(h)
ex = np.array([exact_rate(v) for v in h])

assert np.allclose(fc, -(949.25 + 25 * h**2), rtol=0, atol=1e-8)
assert np.allclose(ex, -(949.25 + 74 * h**2 + 4 / 3 * h**4), rtol=0, atol=1e-8)
check("face-center rate, h = 0.5", fc[0], -955.50, 1e-9)
check("exact rate, h = 0.5", ex[0], -967.8333, 5e-5)
check("face-center rate, h = 0.001", fc[-1], -949.25, 1e-4)
slope = np.polyfit(np.log(h), np.log(np.abs(fc - m1_rate)), 1)[0]
check("observed convergence order", slope, 2.0, 1e-3)

# Line plots: RiskPlot's line charts (TimeSeriesRiskPlot) require a datetime axis, so matplotlib.
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))
side = 2 * h
ax1.axhline(m1_rate, color="black", ls="--", lw=1.2, label="point divergence  −949.25")
ax1.plot(side, fc, "o-", color=IN_COLOR, ms=4, label="face-center flux")
ax1.plot(side, ex, "s-", color=OUT_COLOR, ms=3.5, alpha=0.8, label="exact face integrals")
ax1.annotate("−955.50", (1, fc[0]), xytext=(0.3, -956.8), color=IN_COLOR,
             arrowprops=dict(arrowstyle="->", color=IN_COLOR))
ax1.annotate("−967.83", (1, ex[0]), xytext=(0.3, -967.3), color=OUT_COLOR,
             arrowprops=dict(arrowstyle="->", color=OUT_COLOR))
ax1.set(xscale="log", xlabel="box side length  2h  [cm]", ylabel=r"$\partial\theta/\partial t$  [hr$^{-1}$]",
        title="Estimates converge to the point divergence")
ax1.invert_xaxis()
ax1.legend(loc="center right")
ax1.grid(alpha=0.3)

ax2.loglog(side, np.abs(fc - m1_rate), "o-", color=IN_COLOR, ms=4, label=r"face-center: $25h^2$")
ax2.loglog(side, np.abs(ex - m1_rate), "s-", color=OUT_COLOR, ms=3.5, alpha=0.8,
           label=r"exact: $74h^2 + \frac{4}{3}h^4$")
ax2.loglog(side, 5 * side**2, color="gray", ls=":", label="slope 2 reference")
ax2.set(xlabel="box side length  2h  [cm]", ylabel=r"|estimate − point divergence|  [hr$^{-1}$]",
        title=f"Second-order convergence (fitted slope {slope:.3f})")
ax2.invert_xaxis()
ax2.legend(loc="lower left")
ax2.grid(alpha=0.3, which="both")
fig.tight_layout()
plt.show()

## 12. Summary

| Method | $\partial\theta/\partial t$ [hr⁻¹] | $Q_\text{in}$ [cm³/hr] | $Q_\text{out}$ [cm³/hr] |
|---|---:|---:|---:|
| 1. Point divergence at (2.5, 3.5) | −949.25 | — | — |
| 2. Face-center flux (midpoint rule) | −955.50 | 770.50 | 1726.00 |
| 3. Exact face integrals | −967.8333 | 778.8333 | 1746.6667 |

All three agree on the $x$-contribution (−32.50 hr⁻¹). The spread comes entirely from the $4x^2y^3$ term in $q_y$. The exact
answer equals the volume integral of $\nabla\cdot\mathbf q$ to the last digit, as the divergence theorem requires.

In [ ]:
print("All assertions passed.")